<font color='#007CE5'>**LIMPIEZA DE ARTÍCULOS**</font>

In [1]:
import pandas as pd
import numpy as np
from rapidfuzz import fuzz
import re
from urllib.parse import urlparse
import tldextract

# pd.options.display.max_rows = None

df = pd.read_csv("csv_articulos_unidos.csv", sep=",")

df

,Título,Enlace,Fecha,texto,idioma,clase,provincia,ciudad
0,Els Mossos investiguen l'apunyalament de quatr...,https://www.elperiodico.cat/ca/successos/20200...,05/01/2020,Els Mossos investiguen l'apunyalament de quatr...,ca,Lesions,Barcelona,Barcelona
1,Detienen la propietaria de un bar y su corredo...,https://www.lavanguardia.com/local/catalunya/2...,08/01/2020,Detienen la propietaria de un bar y su corredo...,es,Estafes,Tarragona,Cunit
2,La mare menor d'edat del nadó que el seu pare ...,https://www.ara.cat/societat/llancat-besos-dem...,09/01/2020,La mare menor d'edat del nadó que el seu pare ...,ca,No és delicte,NaN,NaN
3,"Un nen de 12 anys de Badalona, segrestat des d...",https://www.ara.cat/societat/segrest-badalona-...,10/01/2020,"Un nen de 12 anys de Badalona, segrestat des d...",ca,Segrest,Barcelona,Badalona
4,El figurant d'una roda de reconeixement va aca...,https://www.elperiodico.cat/ca/successos/20200...,11/01/2020,El figurant d'una roda de reconeixement va aca...,ca,No és delicte,NaN,NaN
...,...,...,...,...,...,...,...,...
2790,La Guàrdia Civil difon el retrat robot de l'ho...,https://www.ara.cat/societat/guardia-civil-ret...,07/03/2020,La Guàrdia Civil difon el retrat robot de l'ho...,ca,Assassinat Consumat,Tarragona,Tortosa
2791,"Els delictes augmenten un 2,9% a Barcelona el ...",https://www.ara.cat/societat/delictes-augmente...,27/01/2020,"Els delictes augmenten un 2,9% a Barcelona el ...",ca,Furts,Barcelona,Barcelona
2792,Mor el periodista de l'Ajuntament de Barcelona...,https://www.ara.cat/societat/mor-ajuntament-ba...,22/01/2020,Mor el periodista de l'Ajuntament de Barcelona...,ca,Assassinat Consumat,Barcelona,Barcelona
2793,"""Era una mort anunciada""",https://www.ara.cat/societat/mort-anunciada_1_...,16/01/2020,"""Era una mort anunciada"" TarragonaEls treballa...",ca,Homici per imprudència,Tarragona,Tarragona


In [2]:
df['id'] = range(1, len(df) + 1)

cols = ["id"] + [col for col in df.columns if col != "id"]
df = df[cols]

In [3]:
df

,id,Título,Enlace,Fecha,texto,idioma,clase,provincia,ciudad
0,1,Els Mossos investiguen l'apunyalament de quatr...,https://www.elperiodico.cat/ca/successos/20200...,05/01/2020,Els Mossos investiguen l'apunyalament de quatr...,ca,Lesions,Barcelona,Barcelona
1,2,Detienen la propietaria de un bar y su corredo...,https://www.lavanguardia.com/local/catalunya/2...,08/01/2020,Detienen la propietaria de un bar y su corredo...,es,Estafes,Tarragona,Cunit
2,3,La mare menor d'edat del nadó que el seu pare ...,https://www.ara.cat/societat/llancat-besos-dem...,09/01/2020,La mare menor d'edat del nadó que el seu pare ...,ca,No és delicte,NaN,NaN
3,4,"Un nen de 12 anys de Badalona, segrestat des d...",https://www.ara.cat/societat/segrest-badalona-...,10/01/2020,"Un nen de 12 anys de Badalona, segrestat des d...",ca,Segrest,Barcelona,Badalona
4,5,El figurant d'una roda de reconeixement va aca...,https://www.elperiodico.cat/ca/successos/20200...,11/01/2020,El figurant d'una roda de reconeixement va aca...,ca,No és delicte,NaN,NaN
...,...,...,...,...,...,...,...,...,...
2790,2791,La Guàrdia Civil difon el retrat robot de l'ho...,https://www.ara.cat/societat/guardia-civil-ret...,07/03/2020,La Guàrdia Civil difon el retrat robot de l'ho...,ca,Assassinat Consumat,Tarragona,Tortosa
2791,2792,"Els delictes augmenten un 2,9% a Barcelona el ...",https://www.ara.cat/societat/delictes-augmente...,27/01/2020,"Els delictes augmenten un 2,9% a Barcelona el ...",ca,Furts,Barcelona,Barcelona
2792,2793,Mor el periodista de l'Ajuntament de Barcelona...,https://www.ara.cat/societat/mor-ajuntament-ba...,22/01/2020,Mor el periodista de l'Ajuntament de Barcelona...,ca,Assassinat Consumat,Barcelona,Barcelona
2793,2794,"""Era una mort anunciada""",https://www.ara.cat/societat/mort-anunciada_1_...,16/01/2020,"""Era una mort anunciada"" TarragonaEls treballa...",ca,Homici per imprudència,Tarragona,Tarragona


In [4]:
null_df = df.isnull().sum()
null_df

df[df["texto"].isnull()]

df.loc[df["texto"].isnull(), "texto"] = df.loc[df["texto"].isnull(), "Título"]

In [5]:
datosclase = sorted([str(x) for x in df["clase"].unique()])

for i in range(len(datosclase)):
    for j in range(i+1, len(datosclase)):
        score = fuzz.ratio(datosclase[i], datosclase[j])
        if score > 85:
            print(f"{score:.1f}% - '{datosclase[i]}' vs '{datosclase[j]}'")

99.0% - 'Acusació,  denúncia falsa i simulació de delictes' vs 'Acusació, denúncia falsa i simulació de delictes'
85.7% - 'Contra els drets dels treballadors' vs 'Delictes contra els drets dels treballadors'
87.2% - 'Contra els drets i deures familiars' vs 'Delicte contra els drets i deures familiars'
88.9% - 'Furt' vs 'Furts'
95.7% - 'Homici per imprudència' vs 'Homicidi per imprudència'
94.1% - 'Homicidi Consumat' vs 'Homicidi consumat'


In [6]:
df["clase"] = df["clase"].replace({
    'Delictes contra els drets dels treballadors': 'Contra els drets dels treballadors',
    'Acusació,  denúncia falsa i simulació de delictes': 'Acusació, denúncia falsa i simulació de delictes',
    'Furts': 'Furt',
    'Homicidi consumat': 'Homicidi Consumat',
    'Homici per imprudència': 'Homicidi per imprudència',
    'Delicte contra els drets i deures familiars': 'Contra els drets i deures familiars',
    'No és delicte': 'Accident / Desaparició'
}, regex=True)


df["clase"].unique()

array(['Lesions', 'Estafes', 'Accident / Desaparició', 'Segrest',
       'Homicidi per imprudència', 'Homicidi Consumat',
       'Assassinat Consumat', 'Contra la salut pública',
       'Blanqueig de capitals', 'Robatori amb violència i/o intimidació',
       'Agressions sexuals i Abusos sexuals',
       'Contra els recursos naturals i el medi ambient',
       'Contra la llibertat sexual',
       'Contra exercici drets fonamentals i llibertats públiques', 'Furt',
       'Robatori amb força', 'Robatori amb força interior vehicle',
       "Atemptat a autoritat, agents de l'autoritat i resistència i desobediència",
       'Amenaces', 'Rebel·lió', 'Organitzacions i grups criminals',
       'Desordres públics', 'Homicidi Temptativa',
       'Falsedats documentals', 'Detenció il·legal',
       "Conduir sota els efectes d'alcohol i drogues",
       'Propietat industrial', 'Ocupació immobles', 'Incendi', 'Danys',
       "Altres delictes contra l'ordre públic",
       'Faltes contra els interes

In [8]:
datosubicacion = sorted([str(x) for x in df["ciudad"].unique()])

for i in range(len(datosubicacion)):
    for j in range(i+1, len(datosubicacion)):
        score = fuzz.ratio(datosubicacion[i], datosubicacion[j])
        if score > 85:
            print(f"{score:.1f}% - '{datosubicacion[i]}' vs '{datosubicacion[j]}'")



91.4% - 'El Pont de Vilomara' vs 'Pont de Vilomara'
91.3% - 'Franqueses del Vallès' vs 'Les Franqueses del Vallès'
97.0% - 'Molet del Vallès' vs 'Mollet del Vallès'
85.7% - 'Mont-roig del Camp' vs 'Montbrió del Camp'
85.7% - 'Ripoll' vs 'Ripollet'
87.0% - 'Roda de Berà' vs 'Roda de Ter'
95.8% - 'Santa Coloma de Gramanet' vs 'Santa Coloma de Gramenet'


In [9]:
df["ciudad"] = df["ciudad"].replace({
    'L’Hospitalet de Llobregat': "L'Hospitalet de Llobregat",
    "l'Hospitalet de Llobregat": "L'Hospitalet de Llobregat",
    'Les Lloses': 'Les Llosses',
    'Santa Coloma de Gramenet': 'Santa Coloma de Gramanet',
    'Pont de Vilomara': 'El Pont de Vilomara',
    'Les Les Franqueses del Vallès': 'Franqueses del Vallès',
    'Les Franqueses del Vallès': 'Franqueses del Vallès',
    'Franqueses del Vallès': 'Les Franqueses del Vallès',
    'Molet del Vallès': 'Mollet del Vallès'
}, regex=True)

df["ciudad"].unique()

array(['Barcelona', 'Cunit', nan, 'Badalona', 'Girona', 'Tarragona',
       'Terrassa', 'Sant Joan Despí', 'Sabadell', 'Mont-roig del Camp',
       'Santa Coloma de Gramanet', 'La Selva', 'Cubelles',
       'Santa Coloma de Farners', 'Figueres', 'Igualada', 'Amposta',
       'El Vendrell', 'Catalunya', 'Reus', 'Castellar del Vallès',
       'Sant Cugat del Vallès', 'Corbera de Llobregat',
       'Esplugues de Llobregat', 'Rosselló', 'Premià de Mar',
       'Castellví de Rosanes', 'Cambrils', 'Sant Feliu de Guíxols',
       'El El Pont de Vilomara', 'Sant Adrià de Besòs', 'Pineda de Mar',
       'Manresa', 'Mollet del Vallès', 'Vallirana', 'Sant Feliu Sasserra',
       'Calafell', 'Les Borges Blanques', 'Palafrugell',
       'Sant Pere de Vilamajor', 'Botarell', 'La Jonquera',
       'Sant Quirze del Vallès', 'Garrotxa', "L'Hospitalet de Llobregat",
       'Salou', 'Lleida', 'Caldes de Malavella', 'El Masnou',
       'Llinars del Vallès', 'Pallejà', 'Sant Pol de Mar', 'Vila-seca',
     

In [ ]:
import re

def extraer_medio(url, base_host=None):
    if pd.isna(url):
        return pd.NA
    s = str(url).strip()

    if s.startswith("//"):
        s = "http:" + s

    first_seg = s.split("/", 1)[0]
    if not re.match(r"^\w+://", s) and "." in first_seg:
        s = "http://" + s 

    
    parsed = urlparse(s)
    host = parsed.netloc

    if not host:
        if base_host:
            host = urlparse(base_host if re.match(r"^\w+://", base_host) else "http://" + base_host).netloc
        else:
            return pd.NA

    host = host.split(":")[0]

    try:
        ext = tldextract.extract(host)
        return ext.domain or pd.NA
    except Exception:

        comunes = {"www", "m", "amp", "es", "en", "fr"}
        partes = [p for p in host.split(".") if p]
        partes = [p for p in partes if p.lower() not in comunes]
        if len(partes) >= 2:
            return partes[-2]
        elif partes:
            return partes[-1]
        return pd.NA

df["medio"] = df["Enlace"].apply(extraer_medio)

In [11]:
datosmedio = sorted([str(x) for x in df["medio"].unique()])

df["medio"] = df["medio"].replace({
    r'^larazon$': 'La Razón',
    r'^ara$': 'Ara',
    r'^elperiodico$': 'El Periódico',
    r'^lavanguardia$': 'La Vanguardia'
}, regex=True)

df["medio"].unique()

array(['El Periódico', 'La Vanguardia', 'Ara', 'La Razón'], dtype=object)

In [ ]:
# df.to_csv(r'C:\Users\Usuario\Desktop\2. DATOS\Proyecto\Crimenes\articulos.csv', sep=';')

In [12]:
# num_de_clases = df.groupby("clase")["clase"].count().sort_values(ascending=False)
num_de_clases = df[~df["clase"].isin(["Accident / Desaparició"])].groupby("clase")["clase"].count().sort_values(ascending=False)
num_de_clases

clase
Assassinat Consumat                    290
Lesions                                216
Homicidi Consumat                      208
Agressions sexuals i Abusos sexuals    202
Contra la salut pública                166
                                      ... 
Faltes contra l'ordre públic             1
Hisenda pública i seguretat social       1
Injúria                                  1
Mercat i consumidors                     1
Usurpació                                1
Name: clase, Length: 83, dtype: int64

<font color='#007CE5'>**CREACIÓN DE CUADRO PARA WORDCLOUD**</font>

In [ ]:
import pandas as pd
import re
import unicodedata
from stopwordsiso import stopwords as stopwords_iso


def quitar_acentos(texto):
    return ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )


stopwords = set(STOPWORDS) 
stopwords.update(stopwords_iso("es")) 
stopwords.update(stopwords_iso("ca"))

stopwords.update(["mossos", "esquadra", "ara", "davant", "suceso", "policia"])


def limpiar_stopwords(texto):
    texto = str(texto).lower()
    texto = quitar_acentos(texto)
    texto = re.sub(r"[^a-zA-Záéíóúñçàèìòù ]", " ", texto)
    palabras = [w for w in texto.split() if w not in stopwords]

    return " ".join(palabras)

df["texto_limpio"] = df["texto"].apply(limpiar_stopwords)


df

,id,Título,Enlace,Fecha,texto,idioma,clase,provincia,ciudad,medio,texto_limpio
0,1,Els Mossos investiguen l'apunyalament de quatr...,https://www.elperiodico.cat/ca/successos/20200...,05/01/2020,Els Mossos investiguen l'apunyalament de quatr...,ca,Lesions,Barcelona,Barcelona,El Periódico,investiguen apunyalament quatre persones bcn m...
1,2,Detienen la propietaria de un bar y su corredo...,https://www.lavanguardia.com/local/catalunya/2...,08/01/2020,Detienen la propietaria de un bar y su corredo...,es,Estafes,Tarragona,Cunit,La Vanguardia,detienen propietaria bar corredor aseguradora ...
2,3,La mare menor d'edat del nadó que el seu pare ...,https://www.ara.cat/societat/llancat-besos-dem...,09/01/2020,La mare menor d'edat del nadó que el seu pare ...,ca,Accident / Desaparició,NaN,NaN,Ara,mare menor edat nado pare llancar riu besos de...
3,4,"Un nen de 12 anys de Badalona, segrestat des d...",https://www.ara.cat/societat/segrest-badalona-...,10/01/2020,"Un nen de 12 anys de Badalona, segrestat des d...",ca,Segrest,Barcelona,Badalona,Ara,nen anys badalona segrestat mes hondures barce...
4,5,El figurant d'una roda de reconeixement va aca...,https://www.elperiodico.cat/ca/successos/20200...,11/01/2020,El figurant d'una roda de reconeixement va aca...,ca,Accident / Desaparició,NaN,NaN,El Periódico,figurant roda reconeixement acabar preso undes...
...,...,...,...,...,...,...,...,...,...,...,...
2790,2791,La Guàrdia Civil difon el retrat robot de l'ho...,https://www.ara.cat/societat/guardia-civil-ret...,07/03/2020,La Guàrdia Civil difon el retrat robot de l'ho...,ca,Assassinat Consumat,Tarragona,Tortosa,Ara,guardia civil difon retrat robot home mort tro...
2791,2792,"Els delictes augmenten un 2,9% a Barcelona el ...",https://www.ara.cat/societat/delictes-augmente...,27/01/2020,"Els delictes augmenten un 2,9% a Barcelona el ...",ca,Furt,Barcelona,Barcelona,Ara,delictes augmenten barcelona barcelonano hagut...
2792,2793,Mor el periodista de l'Ajuntament de Barcelona...,https://www.ara.cat/societat/mor-ajuntament-ba...,22/01/2020,Mor el periodista de l'Ajuntament de Barcelona...,ca,Assassinat Consumat,Barcelona,Barcelona,Ara,mor periodista ajuntament barcelona apunyalat ...
2793,2794,"""Era una mort anunciada""",https://www.ara.cat/societat/mort-anunciada_1_...,16/01/2020,"""Era una mort anunciada"" TarragonaEls treballa...",ca,Homicidi per imprudència,Tarragona,Tarragona,Ara,mort anunciada tarragonaels treballadors iqoxe...


In [ ]:
# df.to_csv(r'C:\Users\Usuario\Desktop\2. DATOS\Proyecto\Crimenes\wordcloud.csv', sep=';')